In [9]:
from google.colab import files
uploaded = files.upload()

Saving diabetic_data.csv to diabetic_data.csv


In [10]:
import pandas as pd
import numpy as np

RAW_PATH = "diabetic_data.csv"
CLEANED_PATH = "diabetic_data_cleaned.csv"

pd.set_option("display.max_columns", 60)

## 1. Load the raw dataset

In [11]:
df = pd.read_csv(RAW_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded dataset: 101766 rows, 50 columns


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Explore the data (EDA)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

### Target variable: `readmitted`

In [17]:
print(df["readmitted"].value_counts())
print()
print(df["readmitted"].value_counts(normalize=True).round(3))

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64


The target is **imbalanced** (`NO` ~54%, `>30` ~35%, `<30` ~11%). This matters for Milestone 2 model training (e.g. class weighting, resampling).

### Missing values (this dataset marks missing data as `'?'`, not NaN)

In [18]:
qmark_counts = (df == "?").sum().sort_values(ascending=False)
qmark_counts = qmark_counts[qmark_counts > 0]
missing_pct = (100 * qmark_counts / len(df)).round(1)
pd.DataFrame({"missing_count": qmark_counts, "missing_pct": missing_pct})

,missing_count,missing_pct
weight,98569,96.9
medical_specialty,49949,49.1
payer_code,40256,39.6
race,2273,2.2
diag_3,1423,1.4
diag_2,358,0.4
diag_1,21,0.0


### Duplicate patients

In [19]:
dup = df["patient_nbr"].duplicated().sum()
print(f"{dup} repeat encounters from returning patients")

30248 repeat encounters from returning patients


## 3. Clean the data

Steps:
- Replace `'?'` with proper `NaN`
- Drop columns with too much missingness (`weight` ~97%, `medical_specialty` ~49%, `payer_code` ~40%)
- Drop rows missing `race` (only ~2.2%, safe to drop)
- Remove expired/hospice discharges (can't be "readmitted", so they're noise in the target)
- Keep only the first encounter per patient (avoids leakage from repeat visits later)

In [20]:
df_clean = df.copy()

# Replace '?' with NaN
df_clean.replace("?", np.nan, inplace=True)

# Drop high-missingness columns
cols_to_drop = ["weight", "medical_specialty", "payer_code"]
df_clean.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped columns: {cols_to_drop}")

Dropped columns: ['weight', 'medical_specialty', 'payer_code']


In [21]:
before = len(df_clean)
df_clean.dropna(subset=["race"], inplace=True)
print(f"Dropped {before - len(df_clean)} rows with missing race")

Dropped 2273 rows with missing race


In [22]:
# discharge_disposition_id codes for death/hospice — can't be "readmitted"
expired_codes = [11, 13, 14, 19, 20, 21]

before = len(df_clean)
df_clean = df_clean[~df_clean["discharge_disposition_id"].isin(expired_codes)]
print(f"Removed {before - len(df_clean)} rows for expired/hospice discharges")

Removed 2384 rows for expired/hospice discharges


In [23]:
before = len(df_clean)
df_clean = (
    df_clean.sort_values("encounter_id")
    .drop_duplicates(subset="patient_nbr", keep="first")
)
print(f"Reduced to {len(df_clean)} unique patients (removed {before - len(df_clean)} repeat encounters)")

Reduced to 68167 unique patients (removed 28942 repeat encounters)


In [24]:
print(f"Final cleaned shape: {df_clean.shape}")
df_clean.head()

Final cleaned shape: (68167, 47)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
8,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,68,2,28,0,0,0,398,427,38,8,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,15738,63555939,Caucasian,Female,[90-100),3,3,4,12,33,3,18,0,0,0,434,198,486,8,NaN,NaN,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
10,28236,89869032,AfricanAmerican,Female,[40-50),1,1,7,9,47,2,17,0,0,0,250.7,403,996,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
5,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,31,6,16,0,0,0,414,411,250,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30


## 4. Save the cleaned dataset

In [25]:
df_clean.to_csv(CLEANED_PATH, index=False)
print(f"Saved cleaned dataset to: {CLEANED_PATH}")

Saved cleaned dataset to: diabetic_data_cleaned.csv


## Summary

- Raw dataset: 101,766 rows x 50 columns
- Cleaned dataset: rows/columns after dropping sparse columns, missing-race rows, expired/hospice discharges, and duplicate patients
- `readmitted` target is imbalanced — flag this for the Milestone 2 modeling team
- `diag_1`, `diag_2`, `diag_3` (ICD-9 codes) still need grouping/encoding — that's a Milestone 2 feature engineering task, not done here

**Next step (Milestone 2):** feature engineering + training the readmission risk prediction model on `diabetic_data_cleaned.csv`.